In [1]:
import os
import json
from datetime import datetime
import networkx as nx

# Định nghĩa các đường dẫn tệp tin theo quy ước hệ thống
DATASET_DIR = "dataset"
RESULTS_DIR = "results"

CLUSTER_SUMMARY_PATH = os.path.join(DATASET_DIR, "cluster_summary.json")
ALERTS_SAMPLE_PATH = os.path.join(DATASET_DIR, "alerts_sample.jsonl")
SERVICES_PATH = os.path.join(DATASET_DIR, "services.json")
INCIDENTS_HISTORY_PATH = os.path.join(DATASET_DIR, "incidents_history.json")
OUTPUT_PATH = os.path.join(RESULTS_DIR, "rca_output.json")

# Tự động tạo thư mục chứa kết quả nếu chưa tồn tại
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Đã khởi tạo môi trường và các đường dẫn dữ liệu thành công.")

Đã khởi tạo môi trường và các đường dẫn dữ liệu thành công.


In [2]:
# 1. Tải Service Graph và dựng bằng networkx
with open(SERVICES_PATH, "r", encoding="utf-8") as f:
    services_data = json.load(f)

G = nx.DiGraph()
# Thêm các nút từ danh sách dịch vụ và kho lưu trữ
for svc in services_data.get("services", []):
    G.add_node(svc["name"], type="service", criticality=svc.get("criticality", "medium"))
for store in services_data.get("stores", []):
    G.add_node(store["name"], type="store", criticality=store.get("criticality", "medium"))

# Thêm các cạnh thể hiện luồng gọi (A -> B nghĩa là A GỌI B)
for edge in services_data.get("edges", []):
    G.add_edge(edge["from"], edge["to"], type=edge.get("type"))

# 2. Tải cụm alert cần phân tích (Cluster Summary)
with open(CLUSTER_SUMMARY_PATH, "r", encoding="utf-8") as f:
    cluster_summary = json.load(f)

# 3. Đọc dữ liệu alert thô để lấy chính xác mốc thời gian phát sinh lỗi
alerts_map = {}
if os.path.exists(ALERTS_SAMPLE_PATH):
    with open(ALERTS_SAMPLE_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                alert = json.loads(line)
                a_id = alert.get("alert_id") or alert.get("id")
                alerts_map[a_id] = alert

# 4. Tải lịch sử sự cố (Incident History) phục vụ tầng Retrieval
with open(INCIDENTS_HISTORY_PATH, "r", encoding="utf-8") as f:
    incidents_history = json.load(f)

print(f"Đã tải thành công Topology Graph với {G.number_of_nodes()} nodes và {G.number_of_edges()} edges.")
print(f"Số lượng cụm alert cần xử lý: {len(cluster_summary['clusters'])}")
print(f"Số lượng sự cố lịch sử nạp vào bộ nhớ: {len(incidents_history['incidents'])}")

Đã tải thành công Topology Graph với 14 nodes và 17 edges.
Số lượng cụm alert cần xử lý: 5
Số lượng sự cố lịch sử nạp vào bộ nhớ: 29


In [4]:
def parse_timestamp(ts_str):
    """Hàm bổ trợ parse linh hoạt các định dạng ISO timestamp."""
    for fmt in ("%Y-%m-%dT%H:%M:%SZ", "%Y-%m-%dT%H:%M:%S.%fZ", "%Y-%m-%d %H:%M:%S"):
        try:
            return datetime.strptime(ts_str, fmt)
        except ValueError:
            continue
    return datetime.utcnow()

def calculate_severity_match(cluster_sev, incident_sev):
    """Kiểm tra sự tương đồng mức độ nghiêm trọng giữa cụm hiện tại và lịch sử."""
    c_sev = "critical" if cluster_sev == "crit" else "warning" if cluster_sev == "warn" else cluster_sev.lower()
    i_sev = "critical" if incident_sev == "critical" else "warning" if incident_sev in ("warn", "warning") else incident_sev.lower()
    return c_sev == i_sev

def analyze_cluster_rca(cluster, graph, alerts_db, history_db):
    cluster_services = cluster["services"]
    alert_ids = cluster["alert_ids"]
    cluster_id = cluster["cluster_id"]
    max_severity = cluster["max_severity"]
    
    # ----------------------------------------------------
    # BƯỚC 1: GRAPH TRAVERSAL & PAGERANK SCORING
    # ----------------------------------------------------
    # Trích xuất subgraph chứa các service đang alert
    subgraph = graph.subgraph(cluster_services)
    
    try:
        # Chạy PageRank trên REVERSE GRAPH theo đúng quy tắc vàng của RCA
        pr_scores = nx.pagerank(subgraph.reverse(copy=True), alpha=0.85)
    except Exception:
        # Fallback phân phối đều nếu đồ thị bị cô lập hoặc không tính được PageRank
        pr_scores = {node: 1.0 / len(cluster_services) for node in cluster_services}
        
    max_pr = max(pr_scores.values()) if pr_scores else 1.0
    pagerank_norm = {node: (score / max_pr if max_pr > 0 else 1.0) for node, score in pr_scores.items()}
    
    # ----------------------------------------------------
    # BƯỚC 2: TEMPORAL SCORING (Ưu tiên alert sớm nhất)
    # ----------------------------------------------------
    service_earliest_ts = {}
    for a_id in alert_ids:
        alert_item = alerts_db.get(a_id)
        if alert_item:
            svc = alert_item.get("service")
            ts_str = alert_item.get("timestamp") or alert_item.get("starts_at")
            if svc and ts_str:
                ts_dt = parse_timestamp(ts_str)
                if svc not in service_earliest_ts or ts_dt < service_earliest_ts[svc]:
                    service_earliest_ts[svc] = ts_dt
                    
    # Điền giá trị mặc định từ cluster time_range nếu alert thô thiếu thông tin
    cluster_start_dt = parse_timestamp(cluster["time_range"][0])
    for node in cluster_services:
        if node not in service_earliest_ts:
            service_earliest_ts[node] = cluster_start_dt
            
    all_timestamps = list(service_earliest_ts.values())
    min_ts = min(all_timestamps) if all_timestamps else cluster_start_dt
    max_ts = max(all_timestamps) if all_timestamps else cluster_start_dt
    
    temporal_scores = {}
    time_diff = (max_ts - min_ts).total_seconds()
    for node in cluster_services:
        node_ts = service_earliest_ts.get(node, min_ts)
        if time_diff > 0:
            # Sớm nhất -> 1.0, muộn nhất -> 0.0
            temporal_scores[node] = (max_ts - node_ts).total_seconds() / time_diff
        else:
            temporal_scores[node] = 1.0
            
    # ----------------------------------------------------
    # BƯỚC 3: COMBINED SCORING & CANDIDATE RANKING
    # ----------------------------------------------------
    final_scores = {}
    for node in cluster_services:
        pr_n = pagerank_norm.get(node, 0.0)
        t_s = temporal_scores.get(node, 0.0)
        final_scores[node] = 0.6 * pr_n + 0.4 * t_s
        
    # Sắp xếp và trích xuất Top-3 ứng viên hàng đầu
    sorted_candidates = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)
    graph_top3 = [[node, round(score, 2)] for node, score in sorted_candidates[:3]]
    
    root_cause = sorted_candidates[0][0]
    # Tính confidence score dựa trên tỷ trọng điểm phản ánh độ lệch cấu trúc
    sum_scores = sum(final_scores.values())
    confidence = round(sorted_candidates[0][1] / sum_scores, 2) if sum_scores > 0 else 1.0
    
    # ----------------------------------------------------
    # BƯỚC 4: HISTORICAL INCIDENT RETRIEVAL (kNN Heuristic)
    # ----------------------------------------------------
    matched_incidents = []
    for inc in history_db.get("incidents", []):
        score = 0.0
        
        # 1. Khớp Root Cause Service định danh (+0.4)
        if inc.get("root_cause_service") in cluster_services:
            score += 0.4
            
        # 2. Đo mức độ trùng lặp tập dịch vụ bị ảnh hưởng (+0.2 mỗi node, max +0.4)
        overlap = set(cluster_services).intersection(set(inc.get("services_involved", [])))
        score += min(0.2 * len(overlap), 0.4)
        
        # 3. Khớp mốc độ nghiêm trọng hệ thống (+0.2)
        if calculate_severity_match(max_severity, inc.get("severity", "")):
            score += 0.2
            
        if score >= 0.2:
            matched_incidents.append((inc, score))
            
    # Sắp xếp các sự cố lịch sử tìm được theo điểm tương đồng giảm dần
    matched_incidents.sort(key=lambda x: x[1], reverse=True)
    similar_incident_ids = [item[0]["id"] for item in matched_incidents[:3]]
    
    # ----------------------------------------------------
    # BƯỚC 5: CLASSIFICATION & FALLBACK LOGIC
    # ----------------------------------------------------
    if matched_incidents:
        top_incident, max_score = matched_incidents[0]
        root_cause_class = top_incident["root_cause_class"]
        remediation_str = top_incident["remediation"]
        # Phân tách chuỗi khuyến nghị khắc phục thành mảng danh sách hành động cụ thể
        actions = [act.strip() for act in remediation_str.split('.') if act.strip()]
        method = "graph+retrieval"
        reasoning = (f"Dịch vụ '{root_cause}' được xác định là nguyên nhân gốc thông qua phân tích Topology "
                     f"và chuỗi thời gian cảnh báo. Khớp thành công với sự cố lịch sử {top_incident['id']} "
                     f"đạt độ tương đồng Heuristic là {round(max_score, 2)}.")
    else:
        # Áp dụng cơ chế Fallback an toàn nếu không tìm thấy mẫu sự cố tương đồng
        root_cause_class = "other"
        actions = ["Investigate manually"]
        method = "graph-only-fallback"
        reasoning = (f"Hệ thống phát hiện lỗi tại '{root_cause}' dựa trên luồng lan truyền đồ thị phụ thuộc. "
                     f"Không tìm thấy sự cố lịch sử nào tương thích vững chắc (score >= 0.2), kích hoạt chế độ điều tra thủ công.")
        
    return {
        "cluster_id": cluster_id,
        "graph_top3": graph_top3,
        "root_cause": root_cause,
        "class": root_cause_class,
        "confidence": confidence,
        "actions": actions,
        "reasoning": reasoning,
        "similar_incidents": similar_incident_ids,
        "method": method
    }

In [5]:
output_results = []

# Duyệt qua từng cụm alert được phân nhóm từ cấu trúc dữ liệu đầu vào
for cluster in cluster_summary.get("clusters", []):
    rca_analysis = analyze_cluster_rca(cluster, G, alerts_map, incidents_history)
    output_results.append(rca_analysis)
    
    # In thông tin nhanh ngay tại output của cell để kiểm tra tính đúng đắn
    print(f"📌 Cluster ID: {rca_analysis['cluster_id']} | Max Severity: {cluster['max_severity']}")
    print(f"   ↳ Root Cause Đóng Góp: \033[1;31m{rca_analysis['root_cause']}\033[0m (Confidence: {rca_analysis['confidence']})")
    print(f"   ↳ Loại Lỗi Phân Loại: {rca_analysis['class']} | Phương Pháp: {rca_analysis['method']}")
    print(f"   ↳ Đề Xuất Xử Lý: {rca_analysis['actions'][:2]}\n")

# Đóng gói dữ liệu đầu ra theo đúng JSON Schema quy ước
final_json_payload = {
    "clusters_analyzed": len(cluster_summary.get("clusters", [])),
    "results": output_results
}

# Tiến hành ghi file kết quả rca_output.json
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(final_json_payload, f, indent=2, ensure_ascii=False)

print(f"Đã xuất báo cáo phân tích RCA tự động thành công tại: '{OUTPUT_PATH}'")

📌 Cluster ID: c-000-000 | Max Severity: crit
   ↳ Root Cause Đóng Góp: edge-lb (Confidence: 0.28)
   ↳ Loại Lỗi Phân Loại: connection_pool_exhaustion | Phương Pháp: graph+retrieval
   ↳ Đề Xuất Xử Lý: ['Rollback to v3', '1']

📌 Cluster ID: c-001-000 | Max Severity: warn
   ↳ Root Cause Đóng Góp: recommender-svc (Confidence: 1.0)
   ↳ Loại Lỗi Phân Loại: memory_leak | Phương Pháp: graph+retrieval
   ↳ Đề Xuất Xử Lý: ['Patch leak; rollback v3', '0 trong khi chờ']

📌 Cluster ID: c-002-000 | Max Severity: crit
   ↳ Root Cause Đóng Góp: edge-lb (Confidence: 0.5)
   ↳ Loại Lỗi Phân Loại: ddos | Phương Pháp: graph+retrieval
   ↳ Đề Xuất Xử Lý: ['WAF rate-limit + Cloudflare proxy', 'Geographic rule']

📌 Cluster ID: c-003-000 | Max Severity: crit
   ↳ Root Cause Đóng Góp: checkout-svc (Confidence: 0.5)
   ↳ Loại Lỗi Phân Loại: n_plus_1 | Phương Pháp: graph+retrieval
   ↳ Đề Xuất Xử Lý: ['Batch fetch related products', 'Add monitoring query count / request']

📌 Cluster ID: c-004-000 | Max Severi